In [1]:
import string
import re

import pandas as pd
import numpy as np
import py_stringmatching as sm

from copy import deepcopy

# Similarity
Similarity is a measure of how alike two data items are. It is widely used in tasks such as entity resolution, schema matching, and record linkage.
- Similarity score ∈ [0, 1]
- Distance functions often satisfy: reflexive, symmetric, triangle inequality
- Convert distance to similarity: sim(x, y) = 1 - dist(x, y) or sim(x, y) = 1 / (1 + dist(x, y))


## Utils

In [2]:
def sim_table(TableA:pd.DataFrame, TableB:pd.DataFrame):
    A = pd.DataFrame({"A": TableA.columns})
    B = pd.DataFrame({"B": TableB.columns})
    S = A.assign(key=1).merge(B.assign(key=1), on="key").drop("key", axis=1)
#    S = A.merge(B, how='cross') non funziona con le vecchie versioni
    return S

def random_sim_table(TableA:pd.DataFrame, TableB:pd.DataFrame):
    S = sim_table(TableA, TableB)
    S["sim"] = np.random.rand(len(S))
    return S

def to_sim_table(SimMatrix:pd.DataFrame):
    return SimMatrix.stack().reset_index(name="sim")

def to_sim_matrix(SimTable:pd.DataFrame):
    return SimTable.pivot(index="A", columns="B", values="sim") \
              .rename_axis(None, axis=1).rename_axis(None, axis=0)

def string_preprocess(s:str, char:str=string.punctuation, word:list=[]):
    if type(s) is str:
        s = s.lower()
        for c in char:
            s = s.replace(c, " ")
        for w in word:
            s = s.replace(w, " ")
    else:
        s = str(s)
    s = re.sub(" +", " ", s)
    return s.strip()

## Dataset


In [3]:
SchemaA=['ID', 'Name', 'Vorname', 'Alter']
SchemaB=['No', 'Name', 'First_name', 'Age']
TableA = pd.DataFrame(columns=SchemaA)
TableB = pd.DataFrame(columns=SchemaB)

SimTable=sim_table(TableA,TableB)
SimTable

,A,B
0,ID,No
1,ID,Name
2,ID,First_name
3,ID,Age
4,Name,No
5,Name,Name
6,Name,First_name
7,Name,Age
8,Vorname,No
9,Vorname,Name


In [4]:
SimTable = random_sim_table(TableA, TableB)
SimTable

,A,B,sim
0,ID,No,0.510382
1,ID,Name,0.801601
2,ID,First_name,0.412852
3,ID,Age,0.577953
4,Name,No,0.532165
5,Name,Name,0.342084
6,Name,First_name,0.034033
7,Name,Age,0.359162
8,Vorname,No,0.105537
9,Vorname,Name,0.753585


## Edit-Based Measures
Edit-based measures compute the similarity between two strings by calculating the number of operations needed to transform one string into the other.



In [6]:
# definiamo una funzione generale per il calcolo della similarity
def SimilarityFunction(x, similarity):
  return similarity.get_sim_score(string_preprocess(x['A']), string_preprocess(x['B']))

# e la usiamo per calcolare la tabella di similarità
def LabelBasedSimilarityTable(TableA,TableB, similarity):
  DA=pd.DataFrame({'A': TableA.columns})
  DB=pd.DataFrame({'B': TableB.columns})
  PCC = DA.merge(DB, how='cross')
  PCC.columns=['A','B']
  PCC['sim']=PCC.apply(lambda x: SimilarityFunction(x,similarity), axis=1)
  return PCC.sort_values("sim", ascending=False)

### Levenshtein

Levenshtein Distance (or Edit Distance) measures the dissimilarity of two strings, measuring the minimum number of edits needed to transform one string into the other. Edit operations are the followings:
1. Insert
2. Delete
3. Replace

In [7]:
LabelBasedSimilarityTable(TableA,TableB, sm.Levenshtein())

,A,B,sim
5,Name,Name,1.000000
9,Vorname,Name,0.571429
7,Name,Age,0.500000
10,Vorname,First_name,0.500000
6,Name,First_name,0.400000
15,Alter,Age,0.400000
11,Vorname,Age,0.285714
4,Name,No,0.250000
13,Alter,Name,0.200000
8,Vorname,No,0.142857


### Jaro Distance

Specifically designed for matching names:
1. Search for matching characters within a specific distance (m = number of matching characters)
2. Look for swapped adjacent characters (t = number of transpositions)

In [8]:
LabelBasedSimilarityTable(TableA, TableB, sm.Jaro())

,A,B,sim
5,Name,Name,1.000000
10,Vorname,First_name,0.738095
7,Name,Age,0.722222
15,Alter,Age,0.688889
13,Alter,Name,0.633333
4,Name,No,0.583333
8,Vorname,No,0.547619
2,ID,First_name,0.533333
14,Alter,First_name,0.366667
0,ID,No,0.000000


### Jaro-Winkler Distance

Extension of Jaro considering a common prefix (0 < l < 5) and a costant scaling factor for how much the score is adjusted upwards for having common prefixes (p) 

In [9]:
LabelBasedSimilarityTable(TableA, TableB, sm.JaroWinkler())

,A,B,sim
5,Name,Name,1.000000
10,Vorname,First_name,0.738095
7,Name,Age,0.722222
15,Alter,Age,0.720000
13,Alter,Name,0.633333
4,Name,No,0.625000
8,Vorname,No,0.547619
2,ID,First_name,0.533333
14,Alter,First_name,0.366667
0,ID,No,0.000000


### Hamming Distance
Counts the number of positions with differing characters. Only applicable for strings of equal length.

In [10]:
LabelBasedSimilarityTable(TableA, TableB, sm.HammingDistance())
# doesn't work here beacause only applicable for strings of equal length.

ValueError: Undefined for sequences of unequal length

## Token-Based Measures
Token-based measures work by transforming strings into sets of tokens (words or n-grams) and comparing these sets.


In [20]:
TableA= pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/imdb.csv').astype(str)
TableB= pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/roger_ebert.csv').astype(str)

In [21]:
TableA

,id,movie_name,year,directors,actors,movie_rating,genre,duration
0,0,High-Rise,2015,Ben Wheatley,"Tom Hiddleston, Jeremy Irons, Sienna Miller",6.8,"Action, Drama, Sci-Fi",112 min
1,1,Mercy for Angels,2015,K.C. Amos,"Vida Guerra, Emilio Rivera, John Amos",6.2,"Action, Drama, Thriller",91 min
2,2,Wind Walkers,2015,Russell Friedenberg,"Glen Powell, Zane Holtz, Rudy Youngblood",5.6,"Action, Horror, Thriller",93 min
3,3,Alcatraz Prison Escape: Deathbed Confession,2015,John Edward Lee,"Danny Trejo, Ed O'Ross, Shelby Deekins",6.6,"Action, Crime",93 min
4,4,Barely Lethal,2015,Kyle Newman,"Jaime King, Samuel L. Jackson, Madeleine Stack",5.3,"Action, Adventure, Comedy",96 min
...,...,...,...,...,...,...,...,...
6908,6908,When the Clouds Roll by,1919,Victor Fleming,"Douglas Fairbanks, Albert MacQuarrie, Kathleen...",7.0,"Action, Comedy, Romance",85 min
6909,6909,Tarzan of the Apes,1918,Scott Sidney,"Elmo Lincoln, Enid Markey, True Boardman",6.0,"Action, Adventure",73 min
6910,6910,The Life of General Villa,1914,Christy Cabanne,"Eagle Eye, Robert Harron, Irene Hunt",6.7,"Action, Adventure, Biography",105 min
6911,6911,The Perils of Pauline,1914,Louis J. Gasnier,"Pearl White, Crane Wilbur, Paul Panzer",7.6,Action,199 min


In [22]:
TableB

,id,movie_name,year,directors,actors,critic_rating,genre,pg_rating,duration
0,0,Blade Runner: The Final Cut,1982.0,Ridley Scott,nan,4.0,"Drama, Science Fiction",Rated R,117 minutes
1,1,Ace in the Hole,1951.0,Billy Wilder,"Kirk Douglas,Richard Benedict,Jan Sterling",4.0,"Drama, Film Noir",Rated NR,111 minutes
2,2,Pierrot le Fou,1966.0,nan,"Jean-Paul Belmondo,Anna Karina,Graziella Galvani",2.5,"Drama, Foreign, Indie, Thriller",nan,110 minutes
3,3,Rocket Science,2007.0,nan,"Reece Daniel Thompson,Anna Kendrick,Nicholas D...",3.5,"Comedy, Drama, Indie",Rated R,101 minutes
4,4,Casino Royale,2007.0,Martin Campbell,"Daniel Craig,Eva Green,Judi Dench,Jeffrey Wrig...",4.0,"Action, Adventure, Foreign, Thriller",Rated PG-13,144 minutes
...,...,...,...,...,...,...,...,...,...
3551,3551,My Dinner with Andre,1981.0,nan,nan,4.0,"Comedy, Drama, Indie",Rated PG,110 minutes
3552,3552,Star Wars,1977.0,George Lucas,"Mark Hamill,Carrie Fisher,Harrison Ford,Alec G...",4.0,nan,Rated PG,121 minutes
3553,3553,Dr. Strangelove,1964.0,nan,nan,4.0,"Comedy, Drama, Foreign, War",Rated PG,95 minutes
3554,3554,Belle de Jour,1968.0,Luis Bunuel,"Catherine Deneuve,Jean Sorel,Michel Piccoli,Ge...",4.0,"Drama, Foreign, Indie, Romance",Rated NR,101 minutes


In [15]:
# riportiamo tutto in una funzione
def sim(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, f_similarity):
    return f_similarity.get_raw_score(
            TableA[row["A"]].apply(string_preprocess).tolist(),
            TableB[row["B"]].apply(string_preprocess).tolist()
        )

def ValueOverlapSimilarityTable(TableA:pd.DataFrame, TableB:pd.DataFrame, similarity):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(sim, args=(TableA, TableB, similarity), axis=1)
    return C.sort_values("sim", ascending=False)

### Jaccard Coefficient

Usiamo la funzione [Jaccard](https://anhaidgroup.github.io/py_stringmatching/v0.3.x/Jaccard.html) presente in py_stringmatching. The Jaccard coefficient is widely used for general purpose similarity measure for tokens.


In [23]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.Jaccard())
SimTable

,A,B,sim
0,id,id,0.514393
50,movie_rating,critic_rating,0.074074
30,directors,directors,0.068378
60,genre,genre,0.046512
10,movie_name,movie_name,0.038788
...,...,...,...
32,directors,critic_rating,0.000000
33,directors,genre,0.000000
34,directors,pg_rating,0.000000
35,directors,duration,0.000000


#### Calcolo tramite Join (merge)
Per mostrare che è possibile fare lo stesso calcolo tramite Join:

In [ ]:
TableA = pd.DataFrame({  'AX':  ['prof rossi ugo', 'rossi ugo ing prof', 'verde ugo']})
TableB = pd.DataFrame({ 'AY':   [ 'luigi rossi prof', 'verde ugo', 'ugo rossi ing']})
# 1. Calcolo di tutte le coppie di valori:  prodotto cartesiano 
PCC = TableA.drop_duplicates().assign(key=1).merge(TableB.drop_duplicates().assign(key=1), on="key").drop("key", axis=1)
#PCC = TableA.drop_duplicates().merge(TableB.drop_duplicates(), how='cross')
# si tolgono eventuali duplicati perchè non vogliamo considerarli nel calcolo della Value Overlap
PCC

,AX,AY
0,prof rossi ugo,luigi rossi prof
1,prof rossi ugo,verde ugo
2,prof rossi ugo,ugo rossi ing
3,rossi ugo ing prof,luigi rossi prof
4,rossi ugo ing prof,verde ugo
5,rossi ugo ing prof,ugo rossi ing
6,verde ugo,luigi rossi prof
7,verde ugo,verde ugo
8,verde ugo,ugo rossi ing


In [ ]:
# 2. con la Jaccard semplice l'intersezione viene fatta tramite join esatto, quindi:
INTERSEZIONE =  PCC[PCC.AX==PCC.AY].drop_duplicates() # JOIN ESATTO
INTERSEZIONE

,AX,AY
7,verde ugo,verde ugo


In [ ]:
SoloInAX=PCC.loc[~PCC['AX'].isin(INTERSEZIONE['AX'])][['AX']].drop_duplicates()
SoloInAX

,AX
0,prof rossi ugo
3,rossi ugo ing prof


In [ ]:
SoloInAY=PCC.loc[~PCC['AY'].isin(INTERSEZIONE['AY'])][['AY']].drop_duplicates()
SoloInAY

,AY
0,luigi rossi prof
2,ugo rossi ing


In [ ]:
# 3. la cardinalità di A UNION B  si può calcolare  dalla cardinalità dell'INTERSEZIONE, len(INTERSEZIONE), - come segue
SoloInAX=PCC.loc[~PCC['AX'].isin(INTERSEZIONE['AX'])][['AX']].drop_duplicates()
SoloInAY=PCC.loc[~PCC['AY'].isin(INTERSEZIONE['AY'])][['AY']].drop_duplicates()
print(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))
# e quindi la Value Overlap calcolata con la Jaccard semplice risulta essere
len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

5


0.2

### Overlap Coefficient

The overlap coefficient is a similarity measure related to the Jaccard measure that measures the overlap between two sets, and is defined as the size of the intersection divided by the smaller of the size of the two sets.

In [24]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.OverlapCoefficient())
SimTable

,A,B,sim
0,id,id,1.000000
18,year,id,1.000000
50,movie_rating,critic_rating,0.666667
30,directors,directors,0.192367
60,genre,genre,0.154386
...,...,...,...
32,directors,critic_rating,0.000000
33,directors,genre,0.000000
34,directors,pg_rating,0.000000
35,directors,duration,0.000000


### Dice Coefficient

The Dice similarity score is defined as twice the shared information (intersection) divided by sum of cardinalities

In [25]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.Dice())
SimTable

,A,B,sim
0,id,id,0.679339
50,movie_rating,critic_rating,0.137931
30,directors,directors,0.128003
60,genre,genre,0.088889
10,movie_name,movie_name,0.074680
...,...,...,...
32,directors,critic_rating,0.000000
33,directors,genre,0.000000
34,directors,pg_rating,0.000000
35,directors,duration,0.000000


### Cosine Similarity

Measures angle between two vectors (e.g., term frequency vectors), often used in information retrieval.

In [26]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.Cosine())
SimTable

,A,B,sim
0,id,id,0.717212
50,movie_rating,critic_rating,0.226455
18,year,id,0.164306
30,directors,directors,0.135832
60,genre,genre,0.098160
...,...,...,...
32,directors,critic_rating,0.000000
33,directors,genre,0.000000
34,directors,pg_rating,0.000000
35,directors,duration,0.000000


## Hybrid Measures
Hybrid similarity measures combine edit-based and token-based approaches.



In [35]:
TableA = pd.DataFrame({  'A1':  ['prof rossi ugo', 'rossi ugo ing prof', 'verde ugo']})
TableB = pd.DataFrame({ 'B1':   [ 'luigi rossi prof', 'verde ugo', 'ugo rossi ing']})

### Monge-Elkan
- For each token in X, find max similarity with tokens in Y
\[ ME(x, y) = \frac{1}{|x|} \sum_{x_i \in x} \max_{y_j \in y} sim'(x_i, y_j) \]


In [36]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.MongeElkan())
SimTable

,A,B,sim
0,A1,B1,0.848732



### Soft TF-IDF
- Combines TF-IDF weights and token similarity


In [37]:
SimTable = ValueOverlapSimilarityTable(TableA, TableB, sm.SoftTfIdf())
SimTable

,A,B,sim
0,A1,B1,0.798309


### Extended Jaccard
Nella Extended Jaccard si considera la similarità di ogni coppia del prodotto cartesiano \
calcolata  tramite una  funzione di similarità *interna*; ad esempio la classica Levenshtein

In [38]:
def funzione_similarita_interna(row:pd.Series, f_similarity):
    return f_similarity.get_sim_score(
            string_preprocess(row["A1"]),
            string_preprocess(row["B1"])        )
PCC = TableA.drop_duplicates().assign(key=1).merge(TableB.drop_duplicates().assign(key=1), on="key").drop("key", axis=1)
#PCC = TableA.drop_duplicates().merge(TableB.drop_duplicates(), how='cross')
PCC["sim"] = PCC.apply(funzione_similarita_interna, args={sm.Levenshtein()}, axis=1)
PCC

,A1,B1,sim
0,prof rossi ugo,luigi rossi prof,0.500000
1,prof rossi ugo,verde ugo,0.357143
2,prof rossi ugo,ugo rossi ing,0.571429
3,rossi ugo ing prof,luigi rossi prof,0.444444
4,rossi ugo ing prof,verde ugo,0.222222
5,rossi ugo ing prof,ugo rossi ing,0.277778
6,verde ugo,luigi rossi prof,0.187500
7,verde ugo,verde ugo,1.000000
8,verde ugo,ugo rossi ing,0.153846


In [39]:
#   **Value Overlap** considerando le coppie come sovrapposte (e quindi in INTERSEZIONE) 
#    solo se la loro similarità supera una certa soglia.
#    è una semplice selezione delle righe di PCC
INTERSEZIONE =  PCC[PCC.sim>=0.45]
INTERSEZIONE

,A1,B1,sim
0,prof rossi ugo,luigi rossi prof,0.500000
2,prof rossi ugo,ugo rossi ing,0.571429
7,verde ugo,verde ugo,1.000000


In [42]:
#  stessa "formula" applicata in precedenza per il calcolo della Value Overlap
SoloInAX=PCC.loc[~PCC['A1'].isin(INTERSEZIONE['A1'])][['A1']].drop_duplicates()
SoloInAY=PCC.loc[~PCC['B1'].isin(INTERSEZIONE['B1'])][['B1']].drop_duplicates()
len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

0.75

In [45]:
def funzione_similarita_internaLEV(row:pd.Series): # Levenshtein
    lev = sm.Levenshtein()
    return lev.get_sim_score(
            string_preprocess(row["A1"]),
            string_preprocess(row["B1"])
        )

def extended_value_overlap_sim_LEV(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['A1']
    TY.columns=['B1']
    PCC = TX.drop_duplicates().assign(key=1).merge(TY.drop_duplicates().assign(key=1), on="key").drop("key", axis=1)
#    PCC = TX.drop_duplicates().merge(TY.drop_duplicates(), how='cross')
    PCC["SimJac"] = PCC.apply(funzione_similarita_internaLEV, axis=1)
    INTERSEZIONE =  PCC[PCC.SimJac>=threshold]
    SoloInAX=PCC.loc[~PCC['A1'].isin(INTERSEZIONE['A1'])][['A1']].drop_duplicates()
    SoloInAY=PCC.loc[~PCC['B1'].isin(INTERSEZIONE['B1'])][['B1']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

def value_overlap_extended_jaccard_LEV(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim_ext_LEV"] = C.apply(extended_value_overlap_sim_LEV, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim_ext_LEV",ascending=False)

In [46]:
value_overlap_extended_jaccard_LEV(TableA,TableB,0.45)

,A,B,sim_ext_LEV
0,A1,B1,0.75



### Extendend Jaccard, con funzione interna Jaccard 

Finora abbiamo usato come  funzione di similarità *interna* una funzione di tipo *edit-based*, la classica Levenshtein similarity.

Si possono usare  anche funzioni di similarity di tipo *token-based* come la *Jaccard similarity* 

In [47]:
# Definiamo la funzione interna tramite Jaccard con WhitespaceTokenizer
def funzione_similarita_interna(row:pd.Series):
    jac=sm.Jaccard()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return jac.get_sim_score(
            tok.tokenize(string_preprocess(row["A1"])),
            tok.tokenize(string_preprocess(row["B1"])))

PCC = TableA.drop_duplicates().assign(key=1).merge(TableB.drop_duplicates().assign(key=1), on="key").drop("key", axis=1)
#PCC = TableA.drop_duplicates().merge(TableB.drop_duplicates(), how='cross')
PCC["sim_ext_JAC"] = PCC.apply(funzione_similarita_interna, axis=1)
PCC.head()

,A1,B1,sim_ext_JAC
0,prof rossi ugo,luigi rossi prof,0.50
1,prof rossi ugo,verde ugo,0.25
2,prof rossi ugo,ugo rossi ing,0.50
3,rossi ugo ing prof,luigi rossi prof,0.40
4,rossi ugo ing prof,verde ugo,0.20


In [50]:
def funzione_similarita_internaJaccard(row:pd.Series): # Jaccard
    jac=sm.Jaccard()
    tok = sm.WhitespaceTokenizer(return_set=True)
    return jac.get_sim_score(
            tok.tokenize(string_preprocess(row["A1"])),
            tok.tokenize(string_preprocess(row["B1"])))

def extended_value_overlap_sim_JAC(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['A1']
    TY.columns=['B1']
    PCC = TX.drop_duplicates().assign(key=1).merge(TY.drop_duplicates().assign(key=1), on="key").drop("key", axis=1)
#    PCC = TX.drop_duplicates().merge(TY.drop_duplicates(), how='cross')
    PCC["SimJac"] = PCC.apply(funzione_similarita_internaJaccard, axis=1)
    INTERSEZIONE =  PCC[PCC.SimJac>=threshold]
    SoloInAX=PCC.loc[~PCC['A1'].isin(INTERSEZIONE['A1'])][['A1']].drop_duplicates()
    SoloInAY=PCC.loc[~PCC['B1'].isin(INTERSEZIONE['B1'])][['B1']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))

def value_overlap_extended_jaccard_JAC(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim_ext_JAC"] = C.apply(extended_value_overlap_sim_JAC, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim_ext_JAC",ascending=False)

In [51]:
value_overlap_extended_jaccard_JAC(TableA,TableB,0.45)

,A,B,sim_ext_JAC
0,A1,B1,1.0


In [52]:
TableA= pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/imdb.csv').astype(str)
TableB= pd.read_csv('http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/roger_ebert.csv').astype(str)

In [53]:
value_overlap_extended_jaccard_JAC(TableA,TableB,0.45)

KeyboardInterrupt: 


###  Generalized Jaccard (facoltativo)
 
[http://anhaidgroup.github.io/py_stringmatching/v0.4.1/GeneralizedJaccard.html](http://anhaidgroup.github.io/py_stringmatching/v0.4.1/GeneralizedJaccard.html)


In [ ]:
def generalized_sim(row:pd.Series, TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    j = sm.GeneralizedJaccard(
            sim_func=sm.Levenshtein().get_sim_score,
            threshold=threshold
        )
    return j.get_raw_score(
            TableA[row["A"]].apply(string_preprocess).tolist(),
            TableB[row["B"]].apply(string_preprocess).tolist()
        )

def value_overlap_generalized_jaccard(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim_gen_JAC"] = C.apply(generalized_sim, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim_gen_JAC", ascending=False)

In [ ]:
value_overlap_generalized_jaccard(TableA,TableB,0.45)

,A,B,sim_gen_JAC
0,AX,AY,0.392857


In [ ]:
# Confrontiamole nel nostro esempio
TableA = pd.DataFrame({  'AX':  ['prof rossi ugo', 'rossi ugo ing prof', 'verde ugo']})
TableB = pd.DataFrame({ 'AY':   [ 'luigi rossi prof', 'verde ugo', 'ugo rossi ing']})

# consideriamo qualche attributo in più 
TableA = pd.DataFrame({  'A1':  ['prof rossi ugo', 'rossi ugo ing prof', 'verde ugo'],
                          'A2': ['rossi ugo ing prof','verde ugo', 'bianchi gino']})
TableB = pd.DataFrame({ 'B1':   [ 'luigi rossi prof', 'verde ugo', 'ugo rossi ing'],
                         'B2': ['luigi rossi prof','ugo rossi ing', ''],
                          'B3': ['luigi rossi prof', 'ugo rossi ing', '']})

In [ ]:
LEV = value_overlap_extended_jaccard_LEV(TableA, TableB,0.5)
JAC = value_overlap_extended_jaccard_JAC(TableA, TableB,0.5)
GEN = value_overlap_generalized_jaccard(TableA, TableB,0.5)
LEV.merge(JAC, on=['A','B'], suffixes=('', '_JAC')).merge(GEN, on=['A','B'], suffixes=('', '_GEN'))

,A,B,sim_ext_LEV,sim_ext_JAC,sim_gen_JAC
0,A1,B1,0.75,1.0,0.392857
1,A1,B2,0.40,0.6,0.114286
2,A1,B3,0.40,0.6,0.114286
3,A2,B1,0.20,0.5,0.200000
4,A2,B2,0.00,0.2,0.000000
5,A2,B3,0.00,0.2,0.000000



## Set Similarity Join

Considerato che il metodo Value Overlap deve confrontare in genere moltissimi valori,
quelli contenuti negli attributi dei dataset, il vantaggio di usare come  funzione di similarità *interna* una funzione di tipo *token-based* - in cui la similarità è definita su insiemi di oggetti - risiede nel fatto che il calcolo può essere ottimizzato tramite il concetto di *Set Similarity Join*.

Il **Set Similarity Join** è introddotto nelle slide
https://moodle.unimore.it/pluginfile.php/466188/course/section/30324/Set%20Similarity%20Join_SIWS_2022.pdf


Applichiamo la tecnica del String Similarity Join al nostro contesto utilizzando la seguente libreria disponibile in Magellan;



String Similarity Join: [py_stringsimjoin](http://anhaidgroup.github.io/py_stringsimjoin/v0.1.x/overview.html)

In [54]:
!pip install py_stringsimjoin

  Using cached https://files.pythonhosted.org/packages/52/30/c3807065671c4c780eb3d1859d70ff27d907e282258c78bb65e60fb76778/py-stringsimjoin-0.3.6.tar.gz
  Using cached https://files.pythonhosted.org/packages/10/40/d551139c85db202f1f384ba8bcf96aca2f329440a844f924c8a0040b6d02/joblib-1.3.2-py3-none-any.whl
  Using cached https://files.pythonhosted.org/packages/ab/b3/1f12ebc5009c65b607509393ad98240728b4401bc3593868fb161fdd3760/PyPrind-2.11.3-py2.py3-none-any.whl
  Running setup.py install for py-stringsimjoin ... done
You are using pip version 19.0.3, however version 24.0 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.


In [55]:
import py_stringsimjoin as ssj
import warnings
warnings.filterwarnings('ignore')

In [56]:
def sim__join(row:pd.Series, TableA:pd.DataFrame,TableB:pd.DataFrame, threshold:float):
    TX = TableA[[row["A"]]].applymap(string_preprocess).drop_duplicates()
    TY = TableB[[row["B"]]].applymap(string_preprocess).drop_duplicates()
    TX.columns=['AX']
    TY.columns=['AY']
    
    INTERSEZIONE  = ssj.jaccard_join(     TX, TY, # tabelle su cui effettuare il sim join
                                'AX', 'AY', # chiavi delle tabelle 
                                'AX', 'AY', # attributi di join
                                  sm.WhitespaceTokenizer(return_set=True),
                                  threshold=threshold, 
                                  show_progress=False,
                                  l_out_attrs=['AX'],  r_out_attrs=['AY']
                           )
    SoloInAX=TX.loc[~TX['AX'].isin(INTERSEZIONE['l_AX'])][['AX']].drop_duplicates()
    SoloInAY=TY.loc[~TY['AY'].isin(INTERSEZIONE['r_AY'])][['AY']].drop_duplicates()
    return len(INTERSEZIONE)/(len(SoloInAX)+len(SoloInAY)+len(INTERSEZIONE))


def value_overlap_simjoin_jaccard(TableA:pd.DataFrame, TableB:pd.DataFrame, threshold:float):
    C = sim_table(TableA, TableB)
    C["sim"] = C.apply(sim__join, args=(TableA, TableB, threshold), axis=1)
    return C.sort_values("sim",ascending=False)

In [57]:
value_overlap_simjoin_jaccard(TableA,TableB,0.45)

,A,B,sim
60,genre,genre,0.962228
20,year,year,0.930000
0,id,id,0.514393
50,movie_rating,critic_rating,0.306818
10,movie_name,movie_name,0.249925
...,...,...,...
34,directors,pg_rating,0.000000
35,directors,duration,0.000000
41,actors,critic_rating,0.000000
47,movie_rating,year,0.000000


In [58]:
value_overlap_extended_jaccard_JAC(TableA,TableB,0.45)

KeyboardInterrupt: 

In [59]:
src_links = [
'http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/imdb.csv', 
'http://pages.cs.wisc.edu/~anhai/data/784_data/movies5/csv_files/roger_ebert.csv', 
'http://pages.cs.wisc.edu/~anhai/data/784_data/movies1/csv_files/rotten_tomatoes.csv']

#SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

SOURCES = { i : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

In [60]:
value_overlap_simjoin_jaccard(SOURCES[2],SOURCES[0],0.45).sort_values('sim', ascending=False)

,A,B,sim
93,RatingValue,movie_rating,0.997238
118,Genre,genre,0.994733
18,Year,year,0.897196
87,Duration,duration,0.762626
96,RatingCount,id,0.638889
...,...,...,...
112,Genre,id,0.000000
114,Genre,year,0.000000
115,Genre,directors,0.000000
117,Genre,movie_rating,0.000000


In [ ]:
# l'analogo calcolo con value_overlap_extended_jaccard_JAC non termina ...
#value_overlap_extended_jaccard_JAC(SOURCES[2],SOURCES[0],0.45)

In [61]:
# VERIFICA
# 
TableA = pd.DataFrame({  'AX':  ['prof rossi ugo', 'rossi ugo ing prof', 'verde ugo']})
TableB = pd.DataFrame({ 'AY':   [ 'luigi rossi prof', 'verde ugo', 'ugo rossi ing']})

In [62]:
print("=== Jaccard SimJoin ===")
print(value_overlap_simjoin_jaccard(TableA, TableB, 0.45).sort_values('sim', ascending=False))
print("\n=== Extended Jaccard ===")
print(value_overlap_extended_jaccard_JAC(TableA, TableB, 0.45))

=== Jaccard SimJoin ===
    A   B  sim
0  AX  AY  1.0

=== Extended Jaccard ===
    A   B  sim_ext_JAC
0  AX  AY          1.0


In [63]:
print("=== Jaccard SimJoin ===")
print(value_overlap_simjoin_jaccard(TableA, TableB, 0.75).sort_values('sim', ascending=False))
print("\n=== Extended Jaccard ===")
print(value_overlap_extended_jaccard_JAC(TableA, TableB, 0.75))

=== Jaccard SimJoin ===
    A   B  sim
0  AX  AY  0.5

=== Extended Jaccard ===
    A   B  sim_ext_JAC
0  AX  AY          0.5
